# 🏏 IPL Web Scraper Notebook
Scrapes **IPL squads**, **bowling styles**, and **fixtures** from ESPNcricinfo / Cricbuzz / Wikipedia.
Saves everything to `ipl_squads.csv`, `ipl_squads.json`, `ipl_fixtures.csv`, `ipl_fixtures.json`.

**This notebook works standalone — no changes needed to your model notebooks.**
Just run all cells and the CSVs will be ready to use.

In [ ]:
# ── STEP 0: Install dependencies ─────────────────────────────────
# (run once, comment out afterwards)
# !pip install requests beautifulsoup4 lxml pandas numpy

In [ ]:
# ── STEP 1: Imports ───────────────────────────────────────────────
import sys, os
sys.path.insert(0, '.')    # make sure ipl_scraper.py is in same folder

from ipl_scraper import (
    scrape_squads,
    scrape_fixtures,
    scrape_all,
    normalise_style,
    SPIN_TYPES,
    KNOWN_STYLES,
)
import pandas as pd
import json

print('✅ Imports OK')

In [ ]:
# ── STEP 2: CONFIG ────────────────────────────────────────────────
SEASON      = 2025         # IPL season year
OUTPUT_DIR  = '.'          # where to save CSV/JSON (same folder as notebook)
FORCE       = False        # True = ignore cache, always re-scrape
CSV_FALLBACK = '2024_players_details.csv'   # your existing CSV as backup

print(f'Season: {SEASON} | Output: {OUTPUT_DIR} | Force: {FORCE}')

## Part A — Scrape Squads (player names + bowling styles)

In [ ]:
# ── STEP 3: Scrape squads ─────────────────────────────────────────
squads_df, sq_csv, sq_json = scrape_squads(
    season=SEASON,
    out_dir=OUTPUT_DIR,
    force=FORCE,
    csv_fallback=CSV_FALLBACK,
)

print(f'\nTotal players : {len(squads_df)}')
print(f'Spin bowlers  : {squads_df["longBowlingStyles"].notna().sum()}')
squads_df.head(10)

In [ ]:
# ── STEP 4: Inspect spin bowlers ──────────────────────────────────
spin_bowlers = squads_df[squads_df['longBowlingStyles'].notna()].copy()

print('Spin bowlers by style:\n')
print(spin_bowlers.groupby('longBowlingStyles').size().sort_values(ascending=False).to_string())

print('\nFull spin bowler list:')
spin_bowlers[['Name','Team','longBowlingStyles']].sort_values('Team')

In [ ]:
# ── STEP 5: Inspect squads JSON output ────────────────────────────
with open(sq_json) as f:
    sq_data = json.load(f)

print(f'Scraped at : {sq_data["scraped_at"]}')
print(f'Total      : {sq_data["total_players"]} players')
print(f'Teams      : {list(sq_data["teams"].keys())}')

# Show one team as example
team_example = list(sq_data['teams'].keys())[0]
print(f'\n{team_example} squad:')
for p in sq_data['teams'][team_example]:
    style = p['longBowlingStyles'] or 'pace/unknown'
    print(f"  {p['Name']:<25} {style}")

## Part B — Scrape Fixtures

In [ ]:
# ── STEP 6: Scrape fixtures ───────────────────────────────────────
fixtures_df, fx_csv, fx_json = scrape_fixtures(
    season=SEASON,
    out_dir=OUTPUT_DIR,
    force=FORCE,
)

print(f'\nTotal fixtures: {len(fixtures_df)}')
fixtures_df.head(10)

In [ ]:
# ── STEP 7: Upcoming matches ──────────────────────────────────────
if not fixtures_df.empty:
    upcoming = fixtures_df[fixtures_df['status'].str.contains('scheduled|upcoming|fixture', 
                                                               case=False, na=False)]
    print(f'Upcoming matches: {len(upcoming)}')
    upcoming[['match_no','date','team1','team2','venue','status']].head(10)

In [ ]:
# ── STEP 8: Inspect fixtures JSON ────────────────────────────────
if os.path.exists(fx_json):
    with open(fx_json) as f:
        fx_data = json.load(f)
    print(f'Scraped at      : {fx_data["scraped_at"]}')
    print(f'Total matches   : {fx_data["total_matches"]}')
    print('\nFirst 3 matches:')
    for m in fx_data['matches'][:3]:
        print(f"  Match {m.get('match_no','?'):>3}: {m['team1']} vs {m['team2']} | {m['date']} | {m['venue']}")

## Part C — Use scraped data in your prediction model

In [ ]:
# ── STEP 9: Load scraped squads as replacement for 2024_players_details.csv
# Use this anywhere your notebook loaded the old CSV:

#  OLD:   players = pd.read_csv('2024_players_details.csv')
#  NEW ↓

players = pd.read_csv('ipl_squads.csv')   # ← auto-generated by scraper

# Build spin map (name → style)
spin_map = {
    row['Name']: row['longBowlingStyles']
    for _, row in players.iterrows()
    if pd.notna(row.get('longBowlingStyles'))
    and row['longBowlingStyles'] in SPIN_TYPES
}

print(f'Spin bowlers loaded from ipl_squads.csv: {len(spin_map)}')
for name, style in list(spin_map.items())[:10]:
    print(f'  {name:<28} {style}')

In [ ]:
# ── STEP 10: Get spin bowlers for a specific upcoming match ───────
if not fixtures_df.empty:
    # Pick first upcoming match as example
    match = fixtures_df.iloc[0]
    print(f"Match: {match['team1']} vs {match['team2']}")
    print(f"Venue: {match['venue']}")
    print(f"Date : {match['date']}")
    print()

    # Find spin bowlers in each team
    for team_col in ['team1','team2']:
        team = match[team_col]
        if 'Team' in players.columns:
            team_players = players[players['Team'] == team]
            team_spinners = team_players[team_players['longBowlingStyles'].isin(SPIN_TYPES)]
        else:
            team_spinners = pd.DataFrame()

        print(f'{team} spin bowlers:')
        if not team_spinners.empty:
            for _, r in team_spinners.iterrows():
                print(f'  🌀 {r["Name"]:<25} {r["longBowlingStyles"]}')
        else:
            print('  (team info not available in squad data)')
        print()

In [ ]:
# ── STEP 11: ONE-CLICK refresh — run this cell before any match ───
# This replaces your static CSV with fresh scraped data

print('🔄 Refreshing squad and fixture data...')
results = scrape_all(
    season=SEASON,
    out_dir=OUTPUT_DIR,
    force=True,             # force re-scrape
    csv_fallback=CSV_FALLBACK,
)

print('\n📊 Summary:')
print(f'  Players  : {len(results["squads"]["df"])}')
print(f'  Spin bowlers: {results["squads"]["df"]["longBowlingStyles"].notna().sum()}')
print(f'  Fixtures : {len(results["fixtures"]["df"])}')
print(f'\nFiles saved:')
print(f'  {results["squads"]["csv"]}')
print(f'  {results["squads"]["json"]}')
print(f'  {results["fixtures"]["csv"]}')
print(f'  {results["fixtures"]["json"]}')